# Fetch Player Data from EA Sports API
This cell connects to the EA Sports College Football API and fetches all player data for the CFB 26 national championship roster. It handles pagination to retrieve all players in batches of 100, removes duplicates, and displays the total count of players and columns.

In [ ]:
import time
import requests
import pandas as pd
from math import ceil
from tqdm import tqdm

BASE = "https://drop-api.ea.com/rating/ea-sports-college-football"
HEADERS = {"User-Agent": "Mozilla/5.0"}

LOCALE = "en"
ITERATION = "cfb-26-national-championship"   # change if you want a different roster/iteration
LIMIT = 100

def fetch_page(offset: int):
    params = {
        "locale": LOCALE,
        "limit": LIMIT,
        "iteration": ITERATION,
        "offset": offset,
    }
    r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

# --- first call to get totalItems ---
first = fetch_page(0)
total = first["totalItems"]
pages = ceil(total / LIMIT)

all_rows = []
seen = set()

def add_items(items):
    for it in items:
        pid = it.get("id")
        if pid is None or pid not in seen:
            all_rows.append(it)
            if pid is not None:
                seen.add(pid)

add_items(first["items"])

# --- paginate ---
for page in tqdm(range(1, pages), desc="Downloading pages"):
    offset = page * LIMIT
    payload = fetch_page(offset)
    add_items(payload["items"])
    time.sleep(0.12)  # be polite

df = pd.json_normalize(all_rows, sep="_").drop_duplicates(subset=["id"])
print("Rows:", len(df), "Cols:", df.shape[1], "Expected:", total)

Saved ea_cfb26_all_players.csv
Rows: 11062 Cols: 134 Expected: 11062


## Data Cleaning & Transformation
This cell cleans up the raw API data by removing unnecessary columns (like `_diff` suffixes and empty fields), simplifying stat column names, converting height to a human-readable format (e.g., 6'2"), combining first and last names, and configuring pandas display settings for better visibility.

In [2]:
df = df.drop(columns=[c for c in df.columns if c.endswith("_diff")], errors="ignore")

# Drop duplicate overall column (keep overallRating)
df = df.drop(columns=["stats_overall_value"], errors="ignore")

# Rename stats columns: stats_speed_value -> speed
df = df.rename(columns=lambda c: c.replace("stats_", "").replace("_value", ""))

# Drop placeholder columns
df = df.drop(columns=["weight", "team_isPopular"], errors="ignore")

# Add readable height string (keep height inches + height_str only)
df["height_str"] = (df["height"] // 12).astype(int).astype(str) + "'" + (df["height"] % 12).astype(int).astype(str) + '"'

# Put height and height_str next to each other
cols = df.columns.tolist()
cols.remove("height_str")
h = cols.index("height")
cols.insert(h + 1, "height_str")
df = df[cols]

# Create full name, drop first/last, move name to front
df["name"] = df["firstName"] + " " + df["lastName"]
df = df.drop(columns=["firstName", "lastName"], errors="ignore")
df = df[["name"] + [c for c in df.columns if c != "name"]]

# Display wide (no column cut-off)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

df.head()


,name,id,avatarUrl,height,height_str,overallRating,homeTown,homeState,redShirtStatus,jerseyNum,schoolYear,acceleration,agility,jumping,stamina,strength,awareness,bCVision,blockShedding,breakSack,breakTackle,carrying,catchInTraffic,catching,changeOfDirection,deepRouteRunning,finesseMoves,hitPower,impactBlocking,injury,jukeMove,kickAccuracy,kickPower,kickReturn,leadBlock,manCoverage,mediumRouteRunning,passBlock,passBlockFinesse,passBlockPower,playAction,playRecognition,powerMoves,press,pursuit,runBlock,runBlockFinesse,runBlockPower,runningStyle,shortRouteRunning,spectacularCatch,speed,spinMove,stiffArm,tackle,throwAccuracyDeep,throwAccuracyMid,throwAccuracyShort,throwOnTheRun,throwPower,throwUnderPressure,toughness,trucking,zoneCoverage,conference_id,conference_label,conference_imageUrl,team_id,team_label,team_imageUrl,position_id,position_shortLabel,position_label,position_positionType_id,position_positionType_name,iteration_id,iteration_label
0,Fernando Mendoza,4121,https://ratings-images-prod.pulse.ea.com/colle...,77,"6'5""",99,Miami,Florida,Previous,15,Junior,86,86,83,98,75,99,85,48,83,77,75,33,35,85,31,44,59,48,96,84,33,35,25,48,25,32,45,46,43,99,64,51,31,38,51,44,49,0,32,35,83,76,75,56,89,95,97,88,93,96,99,75,22,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeeh...,38,Indiana,https://drop-assets.ea.com/images/4toZEtAPrEnU...,QB,QB,Quarterback,offense,Offense,cfb-26-national-championship,National Championship
1,Jeremiah Smith,8726,https://ratings-images-prod.pulse.ea.com/colle...,75,"6'3""",98,Miami Gardens,Florida,Eligible,4,Sophomore,96,95,97,93,79,92,94,51,60,84,75,98,95,93,97,44,63,59,96,92,35,36,80,55,62,97,40,39,41,32,66,41,64,70,62,56,55,0,97,99,95,88,75,54,33,37,35,55,43,35,97,72,60,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeeh...,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFY...,WR,WR,Wide Receiver,offense,Offense,cfb-26-national-championship,National Championship
2,Rueben Bain Jr.,22723,https://ratings-images-prod.pulse.ea.com/colle...,75,"6'3""",98,Miami,Florida,Eligible,4,Junior,95,87,82,93,92,98,52,93,33,44,57,46,62,74,46,95,91,87,90,46,24,25,34,55,44,43,51,52,52,33,99,98,40,99,59,55,57,0,43,47,84,45,55,85,33,12,24,24,11,25,93,59,48,ACC,ACC,https://drop-assets.ea.com/images/63IBTgQo7jjk...,52,Miami,https://drop-assets.ea.com/images/4cmIyUOVj1Bd...,RE,REDG,Right Edge,defense,Defense,cfb-26-national-championship,National Championship
3,Caleb Downs,4674,https://ratings-images-prod.pulse.ea.com/colle...,72,"6'0""",97,Hoschton,Georgia,Eligible,2,Junior,95,95,91,99,74,88,77,72,57,65,68,64,75,92,47,61,94,72,97,85,32,32,87,59,87,46,49,48,47,36,92,55,88,98,53,56,55,0,47,79,93,79,75,85,37,43,45,49,48,35,96,71,85,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeeh...,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFY...,FS,FS,Free Safety,defense,Defense,cfb-26-national-championship,National Championship
4,David Bailey,4553,https://ratings-images-prod.pulse.ea.com/colle...,75,"6'3""",97,Irvine,California,Eligible,31,Senior,94,89,85,85,82,99,35,79,34,40,56,41,53,77,37,99,89,75,92,64,25,36,44,56,64,40,67,59,52,39,97,89,55,98,58,52,62,0,37,38,86,55,51,84,35,23,17,21,13,39,92,59,74,Big 12,Big 12,https://drop-assets.ea.com/images/3gwMcSmhfgpv...,98,Texas Tech,https://drop-assets.ea.com/images/2Lxte8K570Vx...,LE,LEDG,Left Edge,defense,Defense,cfb-26-national-championship,National Championship


## Rename Columns & Reorganize Data
This cell renames all columns to user-friendly names (e.g., `overallRating` → `Overall Rating`) and reorganizes them in a logical order: identity info → biographical details → team/conference/position → performance attributes → metadata. The cleaned data is saved to a CSV file.

In [5]:
# ===========================
# 1) FULL RENAME MAP
# ===========================
rename_map = {
    "name": "Name",
    "id": "Player ID",
    "avatarUrl": "Helmet URL",
    "height": "Height (in)",
    "height_str": "Height",
    "overallRating": "Overall Rating",
    "homeTown": "Hometown",
    "homeState": "Home State",
    "redShirtStatus": "Redshirt Status",
    "jerseyNum": "Jersey #",
    "schoolYear": "Class Year",

    # General
    "speed": "Speed",
    "acceleration": "Acceleration",
    "strength": "Strength",
    "agility": "Agility",
    "awareness": "Awareness",
    "jumping": "Jumping",
    "injury": "Injury",
    "stamina": "Stamina",
    "toughness": "Toughness",

    # Ballcarrier
    "carrying": "Carrying",
    "breakTackle": "Break Tackle",
    "trucking": "Trucking",
    "changeOfDirection": "Change of Direction",
    "bCVision": "Ball Carrier Vision",
    "stiffArm": "Stiff Arm",
    "spinMove": "Spin Move",
    "jukeMove": "Juke Move",
    "breakSack": "Break Sack",

    # Blocking
    "runBlock": "Run Block",
    "passBlock": "Pass Block",
    "impactBlocking": "Impact Blocking",
    "runBlockPower": "Run Block Power",
    "runBlockFinesse": "Run Block Finesse",
    "passBlockPower": "Pass Block Power",
    "passBlockFinesse": "Pass Block Finesse",
    "leadBlock": "Lead Block",

    # Passing
    "throwPower": "Throw Power",
    "throwUnderPressure": "Throw Under Pressure",
    "throwAccuracyShort": "Throw Accuracy Short",
    "throwAccuracyMid": "Throw Accuracy Mid",
    "throwAccuracyDeep": "Throw Accuracy Deep",
    "throwOnTheRun": "Throw on the Run",
    "playAction": "Play Action",

    # Defense
    "tackle": "Tackle",
    "powerMoves": "Power Moves",
    "finesseMoves": "Finesse Moves",
    "blockShedding": "Block Shedding",
    "pursuit": "Pursuit",
    "playRecognition": "Play Recognition",
    "manCoverage": "Man Coverage",
    "zoneCoverage": "Zone Coverage",
    "hitPower": "Hit Power",
    "press": "Press",

    # Receiving
    "catching": "Catching",
    "spectacularCatch": "Spectacular Catch",
    "catchInTraffic": "Catch in Traffic",
    "shortRouteRunning": "Short Route Running",
    "mediumRouteRunning": "Medium Route Running",
    "deepRouteRunning": "Deep Route Running",

    # Special Teams
    "kickAccuracy": "Kick Accuracy",
    "kickPower": "Kick Power",
    "kickReturn": "Kick Return",

    # Team / Conference / Position
    "conference_id": "Conference ID",
    "conference_label": "Conference",
    "conference_imageUrl": "Conference Logo URL",

    "team_id": "Team ID",
    "team_label": "Team",
    "team_imageUrl": "Team Logo URL",

    "position_id": "Position ID",
    "position_shortLabel": "Position (Short)",
    "position_label": "Position",
    "position_positionType_id": "Unit ID",
    "position_positionType_name": "Unit",

    # Metadata
    "iteration_id": "Iteration ID",
    "iteration_label": "Iteration",
}

df = df.rename(columns=rename_map)

# ===========================
# 2) ATTRIBUTE CATEGORY ORDER
# ===========================
general = [
    "Speed","Acceleration","Strength","Agility","Awareness",
    "Jumping","Injury","Stamina","Toughness"
]

ballcarrier = [
    "Carrying","Break Tackle","Trucking","Change of Direction","Ball Carrier Vision",
    "Stiff Arm","Spin Move","Juke Move","Break Sack"
]

blocking = [
    "Run Block","Pass Block","Impact Blocking","Run Block Power","Run Block Finesse",
    "Pass Block Power","Pass Block Finesse","Lead Block"
]

passing = [
    "Throw Power","Throw Under Pressure","Throw Accuracy Short","Throw Accuracy Mid",
    "Throw Accuracy Deep","Throw on the Run","Play Action"
]

defense = [
    "Tackle","Power Moves","Finesse Moves","Block Shedding","Pursuit","Play Recognition",
    "Man Coverage","Zone Coverage","Hit Power","Press"
]

receiving = [
    "Catching","Spectacular Catch","Catch in Traffic",
    "Short Route Running","Medium Route Running","Deep Route Running"
]

special_teams = ["Kick Accuracy","Kick Power","Kick Return"]

attr_order = (
    general + ballcarrier + blocking +
    passing + defense + receiving + special_teams
)

# ===========================
# 3) FIXED ORDER (Identity → Bio → Team → Pos → Attr → Meta)
# ===========================
identity = ["Name", "Player ID", "Helmet URL"]

bio = [
    "Height (in)", "Height", "Overall Rating",
    "Jersey #", "Class Year", "Redshirt Status",
    "Hometown", "Home State"
]

team_pos = [
    "Team ID","Team","Team Logo URL",
    "Conference ID","Conference","Conference Logo URL",
    "Position ID","Position (Short)","Position",
    "Unit","Unit ID"
]

meta = ["Iteration ID","Iteration"]

fixed_order = identity + bio + team_pos

# ===========================
# 4) FINAL COLUMN ORDER
# ===========================
fixed_set = set(fixed_order + attr_order + meta)
extras = [c for c in df.columns if c not in fixed_set]  # just in case

new_order = (
    [c for c in fixed_order if c in df.columns] +
    [c for c in attr_order if c in df.columns] +
    extras +
    [c for c in meta if c in df.columns]
)

df = df[new_order]

df.to_csv("ea_cfb26_all_players_clean.csv", index=False)

df.head()

,Name,Player ID,Helmet URL,Height (in),Height,Overall Rating,Jersey #,Class Year,Redshirt Status,Hometown,Home State,Team ID,Team,Team Logo URL,Conference ID,Conference,Conference Logo URL,Position ID,Position (Short),Position,Unit,Unit ID,Speed,Acceleration,Strength,Agility,Awareness,Jumping,Injury,Stamina,Toughness,Carrying,Break Tackle,Trucking,Change of Direction,Ball Carrier Vision,Stiff Arm,Spin Move,Juke Move,Break Sack,Run Block,Pass Block,Impact Blocking,Run Block Power,Run Block Finesse,Pass Block Power,Pass Block Finesse,Lead Block,Throw Power,Throw Under Pressure,Throw Accuracy Short,Throw Accuracy Mid,Throw Accuracy Deep,Throw on the Run,Play Action,Tackle,Power Moves,Finesse Moves,Block Shedding,Pursuit,Play Recognition,Man Coverage,Zone Coverage,Hit Power,Press,Catching,Spectacular Catch,Catch in Traffic,Short Route Running,Medium Route Running,Deep Route Running,Kick Accuracy,Kick Power,Kick Return,runningStyle,Iteration ID,Iteration
0,Fernando Mendoza,4121,https://ratings-images-prod.pulse.ea.com/colle...,77,"6'5""",99,15,Junior,Previous,Miami,Florida,38,Indiana,https://drop-assets.ea.com/images/4toZEtAPrEnU...,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeeh...,QB,QB,Quarterback,Offense,offense,83,86,75,86,99,83,96,98,99,75,77,75,85,85,75,76,84,83,51,45,48,49,44,43,46,48,93,96,97,95,89,88,99,56,51,44,48,38,64,25,22,59,31,35,35,33,32,32,31,33,35,25,0,cfb-26-national-championship,National Championship
1,Jeremiah Smith,8726,https://ratings-images-prod.pulse.ea.com/colle...,75,"6'3""",98,4,Sophomore,Eligible,Miami Gardens,Florida,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFY...,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeeh...,WR,WR,Wide Receiver,Offense,offense,95,96,79,95,92,97,96,93,97,75,84,72,93,94,75,88,92,60,62,40,59,55,56,41,39,55,43,35,35,37,33,55,32,54,41,44,51,70,66,62,60,63,64,95,99,98,97,97,97,35,36,80,0,cfb-26-national-championship,National Championship
2,Rueben Bain Jr.,22723,https://ratings-images-prod.pulse.ea.com/colle...,75,"6'3""",98,4,Junior,Eligible,Miami,Florida,52,Miami,https://drop-assets.ea.com/images/4cmIyUOVj1Bd...,ACC,ACC,https://drop-assets.ea.com/images/63IBTgQo7jjk...,RE,REDG,Right Edge,Defense,defense,84,95,92,87,98,82,90,93,93,57,44,59,74,52,55,45,46,33,59,51,87,57,55,52,52,55,11,25,24,12,33,24,33,85,98,95,93,99,99,44,48,91,40,62,47,46,43,43,46,24,25,34,0,cfb-26-national-championship,National Championship
3,Caleb Downs,4674,https://ratings-images-prod.pulse.ea.com/colle...,72,"6'0""",97,2,Junior,Eligible,Hoschton,Georgia,72,Ohio State,https://drop-assets.ea.com/images/1dUQVKJBqNFY...,Big 10,Big Ten,https://drop-assets.ea.com/images/3MVleaeAJeeh...,FS,FS,Free Safety,Defense,defense,93,95,74,95,88,91,97,99,96,68,65,71,92,77,75,79,85,57,53,49,72,55,56,47,48,59,48,35,45,43,37,49,36,85,55,61,72,98,92,87,85,94,88,75,79,64,47,46,47,32,32,87,0,cfb-26-national-championship,National Championship
4,David Bailey,4553,https://ratings-images-prod.pulse.ea.com/colle...,75,"6'3""",97,31,Senior,Eligible,Irvine,California,98,Texas Tech,https://drop-assets.ea.com/images/2Lxte8K570Vx...,Big 12,Big 12,https://drop-assets.ea.com/images/3gwMcSmhfgpv...,LE,LEDG,Left Edge,Defense,defense,86,94,82,89,99,85,92,85,92,56,40,59,77,35,51,55,64,34,58,67,75,62,52,52,59,56,13,39,17,23,35,21,39,84,89,99,79,98,97,64,74,89,55,53,38,41,37,40,37,25,36,44,0,cfb-26-national-championship,National Championship


## Display Player Data with Images
This cell loads the cleaned CSV file, converts image URL columns (helmet, team logo, conference logo) into HTML image tags, and displays the first 5 rows as an interactive HTML table with rendered images.

In [6]:
from IPython.display import HTML

df = pd.read_csv("ea_cfb26_all_players_clean.csv")

def to_img(url, size=60):
    if pd.isna(url) or url == "":
        return ""
    return f'<img src="{url}" width="{size}">'

image_cols = [
    "Helmet URL",
    "Team Logo URL",
    "https://drop-assets.ea.com/images/3MVleaeAJeehQC9SCckJe3/9ae157ee548e7d69ff13907c87dec94a/Big10.png",
    "Conference Logo URL"
]

for col in image_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: to_img(x, size=60))

# Show only first 5 rows with images
HTML(df.head().to_html(escape=False))


,Name,Player ID,Helmet URL,Height (in),Height,Overall Rating,Jersey #,Class Year,Redshirt Status,Hometown,Home State,Team ID,Team,Team Logo URL,Conference ID,Conference,Conference Logo URL,Position ID,Position (Short),Position,Unit,Unit ID,Speed,Acceleration,Strength,Agility,Awareness,Jumping,Injury,Stamina,Toughness,Carrying,Break Tackle,Trucking,Change of Direction,Ball Carrier Vision,Stiff Arm,Spin Move,Juke Move,Break Sack,Run Block,Pass Block,Impact Blocking,Run Block Power,Run Block Finesse,Pass Block Power,Pass Block Finesse,Lead Block,Throw Power,Throw Under Pressure,Throw Accuracy Short,Throw Accuracy Mid,Throw Accuracy Deep,Throw on the Run,Play Action,Tackle,Power Moves,Finesse Moves,Block Shedding,Pursuit,Play Recognition,Man Coverage,Zone Coverage,Hit Power,Press,Catching,Spectacular Catch,Catch in Traffic,Short Route Running,Medium Route Running,Deep Route Running,Kick Accuracy,Kick Power,Kick Return,runningStyle,Iteration ID,Iteration
0,Fernando Mendoza,4121,,77,"6'5""",99,15,Junior,Previous,Miami,Florida,38,Indiana,,Big 10,Big Ten,,QB,QB,Quarterback,Offense,offense,83,86,75,86,99,83,96,98,99,75,77,75,85,85,75,76,84,83,51,45,48,49,44,43,46,48,93,96,97,95,89,88,99,56,51,44,48,38,64,25,22,59,31,35,35,33,32,32,31,33,35,25,0,cfb-26-national-championship,National Championship
1,Jeremiah Smith,8726,,75,"6'3""",98,4,Sophomore,Eligible,Miami Gardens,Florida,72,Ohio State,,Big 10,Big Ten,,WR,WR,Wide Receiver,Offense,offense,95,96,79,95,92,97,96,93,97,75,84,72,93,94,75,88,92,60,62,40,59,55,56,41,39,55,43,35,35,37,33,55,32,54,41,44,51,70,66,62,60,63,64,95,99,98,97,97,97,35,36,80,0,cfb-26-national-championship,National Championship
2,Rueben Bain Jr.,22723,,75,"6'3""",98,4,Junior,Eligible,Miami,Florida,52,Miami,,ACC,ACC,,RE,REDG,Right Edge,Defense,defense,84,95,92,87,98,82,90,93,93,57,44,59,74,52,55,45,46,33,59,51,87,57,55,52,52,55,11,25,24,12,33,24,33,85,98,95,93,99,99,44,48,91,40,62,47,46,43,43,46,24,25,34,0,cfb-26-national-championship,National Championship
3,Caleb Downs,4674,,72,"6'0""",97,2,Junior,Eligible,Hoschton,Georgia,72,Ohio State,,Big 10,Big Ten,,FS,FS,Free Safety,Defense,defense,93,95,74,95,88,91,97,99,96,68,65,71,92,77,75,79,85,57,53,49,72,55,56,47,48,59,48,35,45,43,37,49,36,85,55,61,72,98,92,87,85,94,88,75,79,64,47,46,47,32,32,87,0,cfb-26-national-championship,National Championship
4,David Bailey,4553,,75,"6'3""",97,31,Senior,Eligible,Irvine,California,98,Texas Tech,,Big 12,Big 12,,LE,LEDG,Left Edge,Defense,defense,86,94,82,89,99,85,92,85,92,56,40,59,77,35,51,55,64,34,58,67,75,62,52,52,59,56,13,39,17,23,35,21,39,84,89,99,79,98,97,64,74,89,55,53,38,41,37,40,37,25,36,44,0,cfb-26-national-championship,National Championship


## Load NFL Combine Data
This cell imports the nflreadpy library and loads NFL combine measurement and performance data from the 2026 season to enable comparison between college player ratings and professional combine metrics.

In [4]:
import nflreadpy as nfl

# Load combine data for specific years (e.g., 2022 and 2023)
# The 'years' parameter expects a list or range of years
combine_data = nfl.load_combine(2026)
combine_data.head(20)

season,draft_year,draft_team,draft_round,draft_ovr,pfr_id,cfb_id,player_name,pos,school,ht,wt,forty,bench,vertical,broad_jump,cone,shuttle
i32,f64,str,f64,f64,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64
2026,null,null,null,null,"""AdamCh01""","""chris-adams-2""","""Chris Adams""","""OT""","""Memphis""","""6-5""",311.0,null,null,null,null,null,null
2026,null,null,null,null,null,"""joey-aguilar-1""","""Jose Aguilar""","""QB""","""Tennessee""","""6-3""",229.0,null,null,null,null,null,null
2026,null,null,null,null,"""AllaDr00""","""drew-allar-1""","""Drew Allar""","""QB""","""Penn St.""","""6-5""",228.0,null,null,null,null,null,null
2026,null,null,null,null,null,"""cj-allen-1""","""CJ Allen""","""LB""","""Georgia""","""6-1""",230.0,null,null,null,null,null,null
2026,null,null,null,null,"""AlleKa01""","""kaytron-allen-1""","""Kaytron Allen""","""RB""","""Penn St.""","""5-11""",216.0,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026,null,null,null,null,"""BartLa01""","""lander-barton-1""","""Lander Barton""","""LB""","""Utah""","""6-5""",233.0,null,null,null,null,null,null
2026,null,null,null,null,"""BeckCa01""","""carson-beck-1""","""Carson Beck""","""QB""","""Miami""","""6-5""",233.0,null,null,null,null,null,null
2026,null,null,null,null,null,"""evan-beerntsen-1""","""Evan Beerntsen""","""G""","""Northwestern""","""6-4""",301.0,5.25,null,32.5,105.0,null,null
